# 27.3 Foundation models & generalist agents

Foundation models replace many narrow systems with one broad prior that can be prompted, adapted, and tooled. This notebook simulates routing, context, tools, and scaling laws without API calls or large models. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(273)


def softmax(x, temperature=1.0):
    z = np.asarray(x, dtype=float) / temperature
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def scaling_loss(C, A=1.2, alpha=0.08):
    return A * np.asarray(C, dtype=float) ** (-alpha)

## The concept, built once (D1)

The scaling sketch $L(C)=A C^{-\alpha}$ gives the lesson losses $L(10^{20})=0.030143$, $L(10^{22})=0.020854$, and $L(10^{24})=0.014427$. A hundredfold compute increase multiplies loss by $100^{-0.08}=0.692$.

In [ ]:
compute_points = np.array([1e20, 1e22, 1e24])
losses = scaling_loss(compute_points)
ratio = 100 ** (-0.08)
transformer_flops = 6 * 70_000_000_000 * 300_000_000_000
mixture_loss = 0.5 * 0.20 + 0.3 * 0.40 + 0.2 * 0.60

assert round(float(losses[0]), 6) == 0.030143
assert round(float(losses[1]), 6) == 0.020854
assert round(float(losses[2]), 6) == 0.014427
assert round(float(ratio), 3) == 0.692
assert transformer_flops == 126000000000000000000000
assert round(mixture_loss, 3) == 0.340

print("scaling losses", np.round(losses, 6))
print("hundredfold ratio", round(float(ratio), 3))
print("70B by 300B rough FLOPs", f"{transformer_flops:.2e}")

A generalist agent is also a router. With logits $[2.0,1.0,-0.5]$ for answer, retrieve, and calculate, the softmax is $[0.690,0.254,0.057]$ and expected cost with costs $[0.2,0.5,0.8]$ is $0.310$.

In [ ]:
def route_and_answer(prompt, context, tools, logits, temperature=1.0, verify=False):
    probs = softmax(logits, temperature=temperature)
    route_names = list(tools.keys())
    route = route_names[int(np.argmax(probs))]
    answer = tools[route](prompt, context)
    verified = True
    if verify and route != "retrieve" and "stale" in context.lower():
        route = "retrieve"
        answer = tools[route](prompt, context)
        verified = False
    expected_cost = float(np.dot(probs, np.array([tools[name].cost for name in route_names])))
    return {"route": route, "answer": answer, "probs": probs, "expected_cost": expected_cost, "verified": verified}

class Tool:
    def __init__(self, cost, fn):
        self.cost = cost
        self.fn = fn

    def __call__(self, prompt, context):
        return self.fn(prompt, context)

def calculate_expression(prompt):
    if "calc:" not in prompt:
        return "no calculation"
    expr = prompt.split("calc:")[-1].strip()
    if "*" in expr:
        left, right = expr.split("*", 1)
        return str(int(left) * int(right))
    if "+" in expr:
        left, right = expr.split("+", 1)
        return str(int(left) + int(right))
    return expr


tools = {
    "answer": Tool(0.2, lambda prompt, context: "parametric answer"),
    "retrieve": Tool(0.5, lambda prompt, context: "retrieved current answer"),
    "calculate": Tool(0.8, lambda prompt, context: calculate_expression(prompt)),
}

route_result = route_and_answer("summarize", "fresh context", tools, np.array([2.0, 1.0, -0.5]))

assert np.round(route_result["probs"], 3).tolist() == [0.69, 0.254, 0.057]
assert round(route_result["expected_cost"], 3) == 0.310

print("route probabilities", np.round(route_result["probs"], 3))
print("expected tool cost", round(route_result["expected_cost"], 3))

## The dataset ladder

The F8 prompt/context ladder rises from one direct prompt to few-shot tasks, distractor context, local tools, and a longer rare-tool setting with calibrated versus overconfident routing.

In [ ]:
def make_foundation_ladder():
    return [
        {"name": "D1 direct prompt", "items": [{"prompt": "summarize", "context": "fresh", "need": "answer"}], "logits": np.array([2.0, 1.0, -0.5]), "complexity": 1},
        {"name": "D2 few-shot skills", "items": [{"prompt": "summarize", "context": "fresh", "need": "answer"}, {"prompt": "calc:2+5", "context": "fresh", "need": "calculate"}, {"prompt": "lookup", "context": "fresh", "need": "retrieve"}], "logits": np.array([1.2, 0.8, 0.7]), "complexity": 2},
        {"name": "D3 distractor context", "items": [{"prompt": "lookup", "context": "stale distractor", "need": "retrieve"}, {"prompt": "summarize", "context": "fresh distractor", "need": "answer"}], "logits": np.array([1.5, 0.9, 0.1]), "complexity": 4},
        {"name": "D4 local tools", "items": [{"prompt": "calc:3*7", "context": "fresh", "need": "calculate"}, {"prompt": "lookup", "context": "stale", "need": "retrieve"}, {"prompt": "summarize", "context": "fresh", "need": "answer"}], "logits": np.array([0.8, 1.0, 1.1]), "complexity": 6},
        {"name": "D5 rare tool routing", "items": [{"prompt": "lookup", "context": "stale rare", "need": "retrieve"}, {"prompt": "calc:8*8", "context": "fresh rare", "need": "calculate"}, {"prompt": "lookup", "context": "stale rare", "need": "retrieve"}, {"prompt": "summarize", "context": "fresh", "need": "answer"}], "logits": np.array([2.7, 0.5, 0.2]), "complexity": 10},
    ]

foundation_ladder = make_foundation_ladder()

for rung in foundation_ladder:
    needs = [item["need"] for item in rung["items"]]
    print(rung["name"], "items", len(rung["items"]), "needs", needs, "complexity", rung["complexity"])

In [ ]:
def run_foundation_rung(rung, temperature=1.0, verify=False):
    correct = 0
    total_cost = 0.0
    traces = []
    for item in rung["items"]:
        result = route_and_answer(item["prompt"], item["context"], tools, rung["logits"], temperature=temperature, verify=verify)
        route = result["route"]
        correct += int(route == item["need"])
        total_cost += result["expected_cost"]
        traces.append((item["prompt"], route, item["need"], result["answer"]))
    capability = correct / len(rung["items"])
    expected_cost = total_cost / len(rung["items"])
    score = capability - 0.15 * expected_cost
    return {"capability": capability, "expected_cost": expected_cost, "score": score, "traces": traces}

foundation_results = []

for rung in foundation_ladder:
    result = run_foundation_rung(rung)
    foundation_results.append(result)
    print(f"{rung['name']}: capability={result['capability']:.3f} expected_cost={result['expected_cost']:.3f} score={result['score']:.3f}")

print("no-tool baseline score", round(1 / 3 - 0.15 * 0.2, 3))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(17, 6))

for i, rung in enumerate(foundation_ladder):
    ax = axes[0, i]
    traces = foundation_results[i]["traces"]
    routes = [trace[1] for trace in traces]
    labels = ["answer", "retrieve", "calculate"]
    counts = [routes.count(label) for label in labels]
    ax.bar(labels, counts)
    ax.set_title(rung["name"])
    ax.tick_params(axis="x", rotation=30)

complexity = [rung["complexity"] for rung in foundation_ladder]
scores = [result["score"] for result in foundation_results]
axes[1, 0].plot(complexity, scores, marker="o")
axes[1, 0].set_title("Score vs context/tool complexity")
axes[1, 0].set_xlabel("complexity")
axes[1, 0].set_ylabel("capability - cost penalty")

for j in range(1, 5):
    probs = softmax(foundation_ladder[j]["logits"])
    axes[1, j].bar(["answer", "retrieve", "calculate"], probs)
    axes[1, j].set_ylim(0, 1)
    axes[1, j].tick_params(axis="x", rotation=30)
    axes[1, j].set_title(f"D{j + 1} route probs")

plt.tight_layout()

## Pitfall on the hardest rung: ignoring router calibration

D5 contains rare retrieve/calculate cases but overconfident answer logits. Temperature calibration and verification against stale context move traffic to tools when the parametric answer is likely wrong.

In [ ]:
d5 = foundation_ladder[-1]
wrong = run_foundation_rung(d5, temperature=0.6, verify=False)
fixed = run_foundation_rung(d5, temperature=2.5, verify=True)

print("overconfident score", round(wrong["score"], 3), "capability", round(wrong["capability"], 3))
print("calibrated verified score", round(fixed["score"], 3), "capability", round(fixed["capability"], 3))
print("fixed traces", fixed["traces"])

## Evaluate it + Practice
- Compare the rung metric against the no-skill baseline printed above.
- Run one cheap sanity check: change one label, threshold, route, or floor and confirm the metric moves in the expected direction.
- Ablate the key idea by turning off the kernel, leak, tool verification, feedback, or safety gate.
- Watch failure signals: flat metric curves, hidden weak coordinates, high cost, or a D5 pitfall that does not improve after the fix.

Practice prompts:
1. Change the D3 noise or shift level and predict which rung changes most.

2. Replace the baseline with a stronger classical or rule-based check and compare the metric.

3. Add one new D5 stress case and explain whether the pitfall fix still works.